# 🔬 Model Explainability (SHAP & LIME)
**Extended · Pattern Recognition for the Rest of Us**

> Black-box models make accurate predictions but give no explanation. Explainability tools let you answer: *why* did the model predict this? Which features mattered? This is essential for trust, debugging, and stakeholder communication.

### What you'll learn
- Global vs local explanations
- SHAP (SHapley Additive exPlanations): theoretically grounded feature attribution
- LIME: local surrogate models for any black box
- Partial Dependence Plots (PDP): marginal feature effects
- How to write a non-technical stakeholder summary

### Dataset: Titanic survival + California Housing
### Time: ~70 minutes

## 🎯 Part 1 — Global Explanations: SHAP Summary

SHAP is based on Shapley values from game theory. For each prediction, SHAP computes each feature's contribution to moving the prediction from the base rate to the actual prediction.

**Properties:**
- **Consistency:** if a feature matters more, its SHAP value is higher
- **Additivity:** SHAP values sum to the prediction (minus base value)
- **Fairness:** credit is fairly distributed across features

For tree models, TreeSHAP computes exact values in polynomial time.

In [ ]:
# SHAP: global importance
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_te)
# For binary classification, shap_values[1] = contribution to class 1 (survived)
shap_survived = shap_values[1] if isinstance(shap_values, list) else shap_values

fig, ax = plt.subplots(figsize=(9, 5))
shap.summary_plot(shap_survived, X_te, feature_names=feat_names, show=False)
plt.title('SHAP Summary Plot — Feature Impact on Survival Prediction')
plt.tight_layout(); plt.show()
print('📌 Each dot is one passenger. Color = feature value (red=high, blue=low)')
print('   X-axis = SHAP value: positive = pushes prediction toward survival')
print('   Top features = most impactful overall')

In [ ]:
# SHAP bar chart: mean absolute impact
shap_importance = pd.Series(np.abs(shap_survived).mean(axis=0), index=feat_names).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(shap_importance.index, shap_importance.values, color='#1e5fa8', edgecolor='white')
for bar, val in zip(bars, shap_importance.values):
    ax.text(val+0.002, bar.get_y()+bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Global Feature Importance (SHAP)')
plt.tight_layout(); plt.show()

## 🔍 Part 2 — Local Explanations: Why THIS Prediction?

Global explanations tell you what matters *on average*. Local explanations explain *one specific prediction*.

In [ ]:
# SHAP force plot for one passenger
passenger_idx = 5  # change to explore different passengers
passenger = X_te[passenger_idx]
pred_prob = rf.predict_proba(passenger.reshape(1,-1))[0,1]
actual = y_te[passenger_idx]

print(f'Passenger features:')
for fname, val in zip(feat_names, passenger):
    print(f'  {fname:<25} {val:.1f}')
print(f'\nActual outcome: {"Survived" if actual else "Did not survive"}')
print(f'Predicted probability of survival: {pred_prob:.3f}')
print()

# Waterfall plot
shap_single = shap_survived[passenger_idx]
fig, ax = plt.subplots(figsize=(9, 5))
base_value = explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value
running = base_value
features_sorted = sorted(zip(shap_single, feat_names), key=lambda x: abs(x[0]), reverse=True)
print(f'Base prediction (average): {base_value:.3f}')
for i, (sv, fn) in enumerate(features_sorted[:7]):
    color = '#1a7a45' if sv > 0 else '#e85d20'
    ax.barh(i, sv, left=running, color=color, edgecolor='white', alpha=0.8)
    ax.text(running + sv/2, i, f'{sv:+.3f}', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    running += sv
ax.set_yticks(range(7)); ax.set_yticklabels([fn for sv,fn in features_sorted[:7]])
ax.axvline(base_value, color='gray', lw=1, ls='--', label=f'Base rate ({base_value:.3f})')
ax.axvline(running, color='black', lw=2, ls='-', label=f'Prediction ({running:.3f})')
ax.set_xlabel('Prediction (probability of survival)')
ax.set_title(f'SHAP Waterfall — Passenger {passenger_idx}: pred={pred_prob:.3f}, actual={"survived" if actual else "died"}')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# LIME: local interpretable model-agnostic explanations
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    X_tr, feature_names=feat_names,
    class_names=['Did not survive','Survived'],
    mode='classification'
)

exp = explainer_lime.explain_instance(
    X_te[passenger_idx],
    rf.predict_proba,
    num_features=7,
    num_samples=1000
)

# Extract LIME explanation
fig, ax = plt.subplots(figsize=(9, 4))
lime_feats = exp.as_list()
lime_feats_sorted = sorted(lime_feats, key=lambda x: abs(x[1]))
labels = [f[0] for f in lime_feats_sorted]
values = [f[1] for f in lime_feats_sorted]
colors = ['#1a7a45' if v>0 else '#e85d20' for v in values]
ax.barh(range(len(labels)), values, color=colors, edgecolor='white')
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=9)
ax.axvline(0, color='black', lw=0.8)
ax.set_title(f'LIME Explanation — Same passenger (positive = pushes toward survival)')
ax.set_xlabel('LIME coefficient')
plt.tight_layout(); plt.show()

In [ ]:
# Partial Dependence Plots
from sklearn.inspection import PartialDependenceDisplay
fig, ax = plt.subplots(figsize=(14, 4))
PartialDependenceDisplay.from_estimator(
    rf, X_te, features=[0,1,2,3],
    feature_names=feat_names,
    kind='average', ax=ax
)
plt.suptitle('Partial Dependence Plots — Average Effect of Each Feature', y=1.02, fontsize=12)
plt.tight_layout(); plt.show()
print('\n📌 PDP shows the average marginal effect of each feature, holding all others constant')

## 📝 Part 3 — Writing a Stakeholder Summary

Translating model explanations into business language is a critical skill. Here's a template:

In [ ]:
# Generate a stakeholder summary memo
print('='*65)
print('TITANIC SURVIVAL PREDICTION — STAKEHOLDER SUMMARY')
print('='*65)
print(f'\nModel: Random Forest ({200} trees) | Test Accuracy: {accuracy_score(y_te,rf.predict(X_te)):.1%}')
print()
print('KEY FINDINGS — What drives survival predictions?')
print('-'*50)
print('1. Sex (female=1): The single strongest predictor. The model')
print('   assigns female passengers significantly higher survival probability,')
print('   consistent with historical "women and children first" protocol.')
print()
print('2. Fare/Passenger Class: Higher-paying passengers in upper classes')
print('   had better survival odds. This likely reflects cabin location')
print('   (proximity to lifeboats) and preferential evacuation.')
print()
print('3. Age: Younger passengers had slightly higher survival rates,')
print('   but the effect is smaller than sex or class.')
print()
print('SAMPLE PREDICTION EXPLANATION')
print('-'*50)
print(f'Passenger {passenger_idx}: {"Female" if passenger[1]==1 else "Male"}, ')
print(f'  Class {int(passenger[0])}, Age {passenger[2]:.0f}, Fare ${passenger[3]:.0f}')
print(f'  Predicted: {pred_prob:.0%} chance of survival')
print(f'  Main factors: Sex (most important), Pclass, Fare')
print()
print('LIMITATIONS')
print('The model reflects historical patterns in this dataset. It should')
print('not be used to make causal claims about future events.')

In [ ]:
# Exercise workspace
# Task 1: Apply SHAP to the California Housing GBM model
housing = fetch_california_housing(as_frame=True)
X_h = housing.data.values; y_h = housing.target.values
X_tr_h,X_te_h,y_tr_h,y_te_h = train_test_split(X_h,y_h,test_size=0.2,random_state=42)
gb = GradientBoostingRegressor(n_estimators=100,random_state=42).fit(X_tr_h,y_tr_h)
# YOUR CODE HERE — shap.TreeExplainer, summary_plot, explain one prediction

# Task 2: Write a 2-paragraph stakeholder memo for the housing model
# What drives house prices? What caveats should a real estate analyst know?
# YOUR CODE HERE

In [ ]:
answers = {
    'q1': '',  # What does a positive SHAP value mean?
    'q2': '',  # What is the difference between SHAP and LIME?
    'q3': '',  # What does a Partial Dependence Plot show?
    'q4': '',  # Why might a black-box model need explainability in a regulated industry?
    'q5': '',  # What is the base value in a SHAP explanation?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print('❌ Still empty: '+str(missing) if missing else '✅ Done! File → Save a copy in GitHub')

## 📚 Further Reading
- [SHAP docs](https://shap.readthedocs.io/)
- [LIME GitHub](https://github.com/marcotcr/lime)
- [Interpretable ML Book (Molnar)](https://christophm.github.io/interpretable-ml-book/)
- [H2O AutoML notebook](h2o-automl.ipynb) — built-in explainability

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*